# PyPI Experiment Econometric Analysis

Econometric analysis of the PyPI downloads experiment with extended time series data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from pathlib import Path

In [ ]:
DATA_DIR = Path("../data")
FIGS_DIR = Path("../figs")
FIGS_DIR.mkdir(exist_ok=True)

INPUT_FILE = DATA_DIR / "pypi_experiment_timeseries_extended.csv"
TREATMENT_DATE = pd.Timestamp("2023-06-04")

## Load data

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["date"])
print(f"Shape: {df.shape}")
print(f"Packages: {df['file_project'].nunique():,}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Days: {df['date'].nunique()}")
df.head()

In [ ]:
print("Treatment distribution:")
print(df.groupby("treatment")["file_project"].nunique())

## 1. Simple Difference-in-Differences

In [ ]:
df["post"] = (df["date"] >= TREATMENT_DATE).astype(int)
df["treatment_x_post"] = df["treatment"] * df["post"]
df["log_downloads"] = np.log1p(df["downloads"])

In [ ]:
X = df[["treatment", "post", "treatment_x_post"]]
X = sm.add_constant(X)
y = df["log_downloads"]

did_model = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": df["file_project"]})
print(did_model.summary())

In [ ]:
did_results = pd.DataFrame({
    "coef": did_model.params,
    "se": did_model.bse,
    "t": did_model.tvalues,
    "p": did_model.pvalues,
    "ci_lower": did_model.conf_int()[0],
    "ci_upper": did_model.conf_int()[1]
})
did_results.to_csv(DATA_DIR / "pypi_experiment_did_results.csv")
did_results

## 2. Panel Fixed Effects

In [ ]:
panel_df = df.set_index(["file_project", "date"])
panel_df = panel_df[["log_downloads", "treatment_x_post", "post"]].dropna()

In [ ]:
fe_model = PanelOLS(
    panel_df["log_downloads"],
    panel_df[["treatment_x_post"]],
    entity_effects=True,
    time_effects=True
).fit(cov_type="clustered", cluster_entity=True)

print(fe_model.summary)

In [ ]:
fe_results = pd.DataFrame({
    "coef": fe_model.params,
    "se": fe_model.std_errors,
    "t": fe_model.tstats,
    "p": fe_model.pvalues
})
fe_results.to_csv(DATA_DIR / "pypi_experiment_fe_results.csv")
fe_results

## 3. Event Study (Daily Treatment Effects)

In [ ]:
event_dummies = pd.get_dummies(df["diff_intervention"], prefix="day")
event_cols = [c for c in event_dummies.columns if c != "day_-1"]

event_df = df.copy()
for col in event_cols:
    event_df[col] = event_dummies[col] * event_df["treatment"]

In [ ]:
panel_event = event_df.set_index(["file_project", "date"])
y_event = panel_event["log_downloads"]
X_event = panel_event[event_cols]

In [ ]:
event_model = PanelOLS(
    y_event,
    X_event,
    entity_effects=True,
    time_effects=True
).fit(cov_type="clustered", cluster_entity=True)

print(f"Number of event study coefficients: {len(event_model.params)}")

In [ ]:
event_results = pd.DataFrame({
    "day": [int(c.replace("day_", "")) for c in event_model.params.index],
    "coef": event_model.params.values,
    "se": event_model.std_errors.values,
    "ci_lower": event_model.params.values - 1.96 * event_model.std_errors.values,
    "ci_upper": event_model.params.values + 1.96 * event_model.std_errors.values
}).sort_values("day")

event_results = pd.concat([
    event_results,
    pd.DataFrame({"day": [-1], "coef": [0], "se": [0], "ci_lower": [0], "ci_upper": [0]})
]).sort_values("day").reset_index(drop=True)

event_results.to_csv(DATA_DIR / "pypi_experiment_event_study_results.csv", index=False)
print(f"Saved {len(event_results)} event study coefficients")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
ax.axvline(x=0, color="red", linestyle="--", linewidth=1, label="Treatment")

ax.fill_between(event_results["day"], event_results["ci_lower"], event_results["ci_upper"], alpha=0.3)
ax.plot(event_results["day"], event_results["coef"], marker="o", markersize=2, linewidth=1)

ax.set_xlabel("Days Since Treatment")
ax.set_ylabel("Treatment Effect (log downloads)")
ax.set_title("Event Study: Daily Treatment Effects on PyPI Downloads")
ax.legend()

plt.tight_layout()
plt.savefig(FIGS_DIR / "pypi_experiment_event_study.png", dpi=150)
plt.show()

## 4. Cumulative Event Study (tt_downloads)

In [ ]:
df["log_tt_downloads"] = np.log1p(df["tt_downloads"])

In [ ]:
cum_event_df = df.copy()
for col in event_cols:
    cum_event_df[col] = event_dummies[col] * cum_event_df["treatment"]

panel_cum = cum_event_df.set_index(["file_project", "date"])
y_cum = panel_cum["log_tt_downloads"]
X_cum = panel_cum[event_cols]

In [ ]:
cum_model = PanelOLS(
    y_cum,
    X_cum,
    entity_effects=True,
    time_effects=True
).fit(cov_type="clustered", cluster_entity=True)

print(f"Number of cumulative event study coefficients: {len(cum_model.params)}")

In [ ]:
cum_results = pd.DataFrame({
    "day": [int(c.replace("day_", "")) for c in cum_model.params.index],
    "coef": cum_model.params.values,
    "se": cum_model.std_errors.values,
    "ci_lower": cum_model.params.values - 1.96 * cum_model.std_errors.values,
    "ci_upper": cum_model.params.values + 1.96 * cum_model.std_errors.values
}).sort_values("day")

cum_results = pd.concat([
    cum_results,
    pd.DataFrame({"day": [-1], "coef": [0], "se": [0], "ci_lower": [0], "ci_upper": [0]})
]).sort_values("day").reset_index(drop=True)

cum_results.to_csv(DATA_DIR / "pypi_experiment_cumulative_event_study_results.csv", index=False)
print(f"Saved {len(cum_results)} cumulative event study coefficients")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.axhline(y=0, color="black", linestyle="-", linewidth=0.5)
ax.axvline(x=0, color="red", linestyle="--", linewidth=1, label="Treatment")

ax.fill_between(cum_results["day"], cum_results["ci_lower"], cum_results["ci_upper"], alpha=0.3)
ax.plot(cum_results["day"], cum_results["coef"], marker="o", markersize=2, linewidth=1)

ax.set_xlabel("Days Since Treatment")
ax.set_ylabel("Treatment Effect (log cumulative downloads)")
ax.set_title("Cumulative Event Study: Treatment Effects on Total PyPI Downloads")
ax.legend()

plt.tight_layout()
plt.savefig(FIGS_DIR / "pypi_experiment_cumulative_event_study.png", dpi=150)
plt.show()

## 5. Hausman Test (FE vs RE)

In [ ]:
re_model = RandomEffects(
    panel_df["log_downloads"],
    panel_df[["treatment_x_post"]]
).fit()

print("Random Effects Model:")
print(re_model.summary)

In [ ]:
b_fe = fe_model.params["treatment_x_post"]
b_re = re_model.params["treatment_x_post"]
var_fe = fe_model.std_errors["treatment_x_post"] ** 2
var_re = re_model.std_errors["treatment_x_post"] ** 2

hausman_stat = (b_fe - b_re) ** 2 / (var_fe - var_re) if var_fe > var_re else np.nan
from scipy import stats
hausman_pval = 1 - stats.chi2.cdf(hausman_stat, 1) if not np.isnan(hausman_stat) else np.nan

print(f"Hausman test statistic: {hausman_stat:.4f}")
print(f"p-value: {hausman_pval:.4f}")
print(f"Conclusion: {'Fixed Effects preferred' if hausman_pval < 0.05 else 'Random Effects acceptable'}")

## 6. Summary Statistics by Treatment Group

In [ ]:
summary_stats = df.groupby(["treatment", "post"]).agg({
    "downloads": ["mean", "median", "std"],
    "tt_downloads": ["mean", "median", "std"],
    "file_project": "nunique"
}).round(2)

summary_stats.columns = ["_".join(col) for col in summary_stats.columns]
summary_stats

## 7. Pre-treatment Parallel Trends Check

In [ ]:
pre_treatment = event_results[event_results["day"] < 0]
print(f"Pre-treatment coefficients (should be ~0 for parallel trends):")
print(f"Mean: {pre_treatment['coef'].mean():.4f}")
print(f"Std: {pre_treatment['coef'].std():.4f}")
print(f"Max absolute: {pre_treatment['coef'].abs().max():.4f}")

significant_pre = pre_treatment[(pre_treatment["ci_lower"] > 0) | (pre_treatment["ci_upper"] < 0)]
print(f"\nSignificant pre-treatment coefficients: {len(significant_pre)} out of {len(pre_treatment)}")